# SVM + RBF acelerado por GPU (cuML/RAPIDS) — combinaciones faltantes

Corre en Google Colab con runtime GPU (T4 gratuita) las combinaciones de
**SVM** que todavía no tienen fila en `results/experiments_master.csv` del
proyecto local, usando **cuML (RAPIDS)** en vez de scikit-learn para
acelerar por hardware el ajuste del kernel RBF (el cuello de botella real,
documentado en `EXPERIMENT_STATUS.md` como el modelo más costoso del
pipeline: O(n²)-O(n³) + `probability=True` agrega 5 fits internos de Platt
scaling por ajuste).

**Esto NO es un modelo distinto.** Reproduce exactamente la misma
metodología que `src/nested_cv.py` + `src/optimization.py` +
`src/balancing.py` + `src/preprocessing.py` del proyecto local:

- Validación cruzada anidada: outer=3 folds, inner=3 folds,
  `StratifiedKFold(shuffle=True, random_state=42)`.
- Preprocesamiento (`OneHotEncoder` + `StandardScaler`) y balanceo
  (SMOTE/ADASYN/class_weight) **dentro** del pipeline, ajustados solo en
  `fit()` de cada fold — nunca en todo el dataset.
- `kernel='rbf'` fijo, `C`/`gamma` en escala log (`C`: 1e-2–1e2, `gamma`:
  1e-4–1e0).
- Los 4 métodos de optimización con los mismos presupuestos:
  `grid_search` (16 combinaciones, 4×4), `random_search` (`n_iter=25`),
  `bayesian_optuna` (Optuna TPE, `n_trials=30`, `seed=42`), `genetic_deap`
  (DEAP: población=16, generaciones=10, crossover uniforme
  `cx_prob=0.6`/`indpb=0.5`, mutación gaussiana `mut_prob=0.3`/`indpb=0.2`,
  torneo `tournament_size=3`, elitismo `elite_size=2`).
- Deduplicación de filas (35 filas, igual que `preprocessing.remove_duplicate_rows`)
  aplicada ANTES de cualquier split, para que los folds de este notebook
  caigan en las mismas particiones que las corridas locales de `svm/none/*`.
- `random_state=42` en todo (folds, samplers, GA).

**Único cambio real:** el paso `"model"` del pipeline es
`cuml.svm.SVC(kernel='rbf', ...)` (GPU) en vez de `sklearn.svm.SVC(...)`
(CPU) — mismo algoritmo (SVM de margen máximo con kernel RBF, solver SMO),
misma interpretación de `C`/`gamma`/`class_weight`. Todo lo demás
(preprocesamiento, balanceo, folds, espacio de búsqueda, operadores del GA,
sampler de Optuna) es el mismo código, transcrito para no depender de los
módulos `src/*` del proyecto (Colab no tiene el repo).

`grid_search` y `random_search` **no** usan `GridSearchCV`/`RandomizedSearchCV`
de scikit-learn envolviendo un pipeline con un estimador de cuML adentro —
se implementan de forma manual (mismo algoritmo: por cada candidato, CV
interna, promedio de folds, se queda con el mejor, refit sobre todo el fold
externo) para no depender de que `sklearn.base.clone()`/`get_params()`/
`set_params()` funcionen sin fricción sobre un estimador de cuML dentro de
un meta-estimador de sklearn. Es un cambio de **implementación**, no de
metodología (mismo algoritmo que `GridSearchCV`/`RandomizedSearchCV` usan
internamente) — mismo principio ya aplicado en el proyecto local para
`bayesian_optuna`/`genetic_deap` (ver docstring de `src/optimization.py`).
`random_search` usa `sklearn.model_selection.ParameterSampler` para generar
los candidatos — es EXACTAMENTE la clase que `RandomizedSearchCV` usa
internamente, así que la secuencia de candidatos muestreados es idéntica
dado el mismo `random_state`.

## ⚠️ Diferencias cuML vs. scikit-learn a tener en cuenta (verificadas, no asumidas)

1. **`probability=True` tuvo un bug real en cuML 25.08** (`AttributeError:
   'NoneType' object has no attribute 'upper'` al convertir de GPU a host)
   — [issue #7205](https://github.com/rapidsai/cuml/issues/7205), arreglado
   por el PR #7207 (ya mergeado). La celda de instalación imprime la
   versión de cuML instalada y una celda posterior (**"Chequeo de
   compatibilidad"**) hace un fit real con `probability=True` +
   `predict_proba` ANTES de correr nada real — si falla, el notebook se
   detiene con un mensaje explícito en vez de bajar silenciosamente a
   `probability=False` o cambiar el scoring.
2. **`class_weight='balanced'` es seguro para este dataset** (target
   binario): el único bug conocido de `class_weight` en SVC de cuML (PR
   #8592, "Implement multiclass SVC with non-uniform class weights") es
   específico de **multiclase** (3+ clases) — no aplica aquí.
3. **`gamma` acepta un `float` directo**, igual que scikit-learn (además de
   `'scale'`/`'auto'`, que este proyecto no usa) — no hace falta adaptar el
   espacio de búsqueda.
4. **`OneHotEncoder` debe forzarse a salida densa** (`sparse_output=False`):
   cuML SVC no garantiza aceptar una matriz dispersa de scipy de la forma
   en que lo hace el SVC de scikit-learn (basado en libsvm). Es un cambio
   de representación (denso vs. disperso), NO cambia los valores
   codificados ni la metodología.
5. **`cache_size`** tiene unidades distintas (`sklearn`: MB; `cuML`: MiB) y
   **`nochange_steps`** es un parámetro de parada temprana del solver SMO
   exclusivo de cuML (sin equivalente en libsvm/sklearn, se deja en su
   default). Ninguno de los dos es un hiperparámetro bajo búsqueda —no
   pueden sesgar la comparación entre los 4 métodos, solo afectan tiempo de
   pared.
6. **Calibración de probabilidad (Platt scaling) es una reimplementación
   propia de cuML**, no litealmente el código de libsvm que usa
   scikit-learn. Las métricas de **etiqueta dura** (`accuracy`,
   `precision`, `recall`, `f1`) dependen del signo de la función de
   decisión, no de la calibración, así que deberían ser muy cercanas a lo
   que darías con CPU para los mismos hiperparámetros — pero `roc_auc`
   podría mostrar pequeñas diferencias atribuibles a la reimplementación de
   la calibración, no a un error de metodología. Documentarlo si se nota
   una diferencia grande.
7. **Versiones de librerías**: Colab instalará lo último que `pip`
   resuelva para `scikit-learn`/`imbalanced-learn`/`optuna`/`deap` (las
   restricciones de los wheels de RAPIDS probablemente no dejan fijar
   exactamente `scikit-learn==1.9.1`/`imbalanced-learn==0.14.2`/
   `optuna==5.0.0`/`deap==1.4.4` como en `requirements.txt` local). La
   celda de imports imprime todas las versiones instaladas — anotarlas
   junto a los resultados.

## Cómo usar este notebook

1. `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)`.
2. Correr las celdas en orden. La celda de instalación de RAPIDS puede
   pedir reiniciar la sesión — si lo hace, reiniciar y seguir desde la
   siguiente celda (no hace falta repetir la instalación).
3. Subir `dataset_final.csv` cuando la celda de carga lo pida.
4. Revisar la celda **"Selección de combinaciones a correr"** — por defecto
   corre las 13 combinaciones de SVM que faltan (ver más abajo cuáles ya
   están corridas localmente). Editar `COMBOS_TO_RUN` para correr un
   subconjunto.
5. Correr la celda de ejecución. Guarda cada fila al CSV apenas termina esa
   combinación (no espera a que terminen las 13) — si Colab se desconecta,
   basta con volver a correr desde la celda de ejecución: salta
   automáticamente lo que ya esté en el CSV de salida.
6. Al final, descargar `svm_gpu_results.csv` y pegar las filas en tu tabla
   maestra local (mismo esquema exacto de columnas).


## 1. Verificar GPU asignada

In [ ]:
!nvidia-smi

## 2. Instalar RAPIDS cuML

Usa el instalador oficial de RAPIDS para Colab (`rapidsai-csp-utils`),
el método recomendado por la documentación de RAPIDS porque ajusta
versiones/paquetes preinstalados de Colab que a veces chocan con un
`pip install cuml-cu12` directo. **Puede pedir reiniciar la sesión** — si
lo hace, reiniciar (`Entorno de ejecución → Reiniciar sesión`) y continuar
desde la celda siguiente sin volver a correr esta.

In [ ]:
!git clone --depth 1 https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

In [ ]:
# Optuna y DEAP no vienen con RAPIDS -- se instalan aparte.
!pip install -q optuna deap

## 3. Imports + chequeo de versiones

In [ ]:
import sys
import time
import json
import random
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

import cuml
from cuml.svm import SVC as cuSVC

import sklearn
import imblearn
import optuna
import deap
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, ParameterGrid, ParameterSampler
from sklearn.metrics import get_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from scipy.stats import loguniform
from deap import base, creator, tools

optuna.logging.set_verbosity(optuna.logging.WARNING)

print("cuml          :", cuml.__version__)
print("scikit-learn  :", sklearn.__version__)
print("imbalanced-learn:", imblearn.__version__)
print("optuna        :", optuna.__version__)
print("deap          :", deap.__version__)
print("numpy         :", np.__version__)
print("pandas        :", pd.__version__)
print()
print("Local (requirements.txt): scikit-learn==1.9.1, imbalanced-learn==0.14.2, "
      "optuna==5.0.0, deap==1.4.4 -- comparar contra lo de arriba y anotar "
      "cualquier diferencia relevante junto a los resultados.")

## 4. Chequeo de compatibilidad cuML (correr ANTES de cualquier combinación real)

Verifica, con datos sintéticos chicos, que:

1. `cuml.svm.SVC(probability=True)` ajusta y `predict_proba` funciona (ver
   aviso #1 más arriba — bug conocido en 25.08).
2. El mismo estimador funciona compuesto dentro de un
   `imblearn.pipeline.Pipeline` (preprocesamiento denso + SMOTE + modelo),
   igual que se usará en las combinaciones reales.
3. `class_weight='balanced'` no rompe nada en el caso binario.

Si cualquiera de estos chequeos falla, el notebook se detiene acá con el
error completo — **no** baja a `probability=False` ni cambia de scoring
por su cuenta.

In [ ]:
from sklearn.datasets import make_classification

X_chk, y_chk = make_classification(
    n_samples=300, n_features=8, n_informative=5, weights=[0.75, 0.25], random_state=42
)
X_chk = X_chk.astype(np.float32)

print("Chequeo 1/3: SVC(probability=True).fit() + predict_proba()...")
clf_chk = cuSVC(kernel="rbf", C=1.0, gamma=0.1, probability=True, random_state=42, output_type="numpy")
clf_chk.fit(X_chk, y_chk)
proba_chk = clf_chk.predict_proba(X_chk)
pred_chk = clf_chk.predict(X_chk)
assert proba_chk.shape == (300, 2), f"forma inesperada de predict_proba: {proba_chk.shape}"
assert np.allclose(proba_chk.sum(axis=1), 1.0, atol=1e-3), "las filas de predict_proba no suman 1"
assert list(clf_chk.classes_) == [0, 1], f"orden de clases inesperado: {clf_chk.classes_}"
print("  OK -- predict_proba shape:", proba_chk.shape, "| classes_:", clf_chk.classes_)

print("Chequeo 2/3: class_weight='balanced' (binario)...")
clf_bal = cuSVC(kernel="rbf", C=1.0, gamma=0.1, probability=True, class_weight="balanced",
                 random_state=42, output_type="numpy")
clf_bal.fit(X_chk, y_chk)
_ = clf_bal.predict_proba(X_chk)
print("  OK")

print("Chequeo 3/3: pipeline completo (preprocesamiento denso + SMOTE + cuML SVC) + CV manual...")
ct_chk = ColumnTransformer(
    transformers=[], remainder=StandardScaler()
)
pipe_chk = ImbPipeline(steps=[
    ("preprocessing", ct_chk),
    ("smote", SMOTE(random_state=42, k_neighbors=3)),
    ("model", cuSVC(kernel="rbf", C=1.0, gamma=0.1, probability=True, random_state=42, output_type="numpy")),
])
cv_chk = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scorer_chk = get_scorer("f1")
fold_scores_chk = []
for tr_idx, val_idx in cv_chk.split(X_chk, y_chk):
    pipe_chk.fit(X_chk[tr_idx], y_chk[tr_idx])
    fold_scores_chk.append(scorer_chk(pipe_chk, X_chk[val_idx], y_chk[val_idx]))
print("  OK -- f1 por fold:", [round(s, 4) for s in fold_scores_chk])

print()
print("Los 3 chequeos pasaron. Es seguro continuar con las combinaciones reales.")

## 5. Subir `dataset_final.csv`

In [ ]:
from google.colab import files

uploaded = files.upload()
assert "dataset_final.csv" in uploaded, (
    f"Se esperaba un archivo llamado 'dataset_final.csv', se subio: {list(uploaded.keys())}"
)
DATASET_PATH = "dataset_final.csv" 

## 6. Cargar, deduplicar y separar X/y

Reproduce exactamente `src/data_loader.load_processed_dataset` +
`src/preprocessing.remove_duplicate_rows` + `src/data_loader.split_features_target`
del proyecto local: mismo dataset -> mismas 35 filas duplicadas eliminadas
(`drop_duplicates(keep="first")`, índice reseteado) -> mismo `X`/`y`. Esto
es necesario para que `StratifiedKFold(shuffle=True, random_state=42)`
genere los mismos folds que usaron las corridas locales de `svm/none/*`.

In [ ]:
TARGET_COLUMN = "default_payment_next_month"
CATEGORICAL_COLUMNS = ["SEX", "EDUCATION", "MARRIAGE"]
RANDOM_STATE = 42

df = pd.read_csv(DATASET_PATH)
print(f"Dataset cargado: {df.shape}")
assert df.shape == (30000, 24), f"forma inesperada: {df.shape} (se esperaba (30000, 24), sin columna ID)"

n_before = len(df)
df = df.drop_duplicates(subset=None, keep="first").reset_index(drop=True)
n_removed = n_before - len(df)
print(f"Filas duplicadas eliminadas: {n_removed}")
assert n_removed == 35, (
    f"Se esperaban 35 filas duplicadas (igual que el dataset local), se encontraron {n_removed}. "
    "Verificar que este sea el MISMO dataset_final.csv que el proyecto local -- si no, "
    "los folds de este notebook NO van a coincidir con los de las corridas locales de svm/none/*."
)

X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]
print(f"X: {X.shape}, balance del target: {y.value_counts(normalize=True).round(4).to_dict()}")

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

## 7. Configuración (idéntica a `src/config.py`)

In [ ]:
N_OUTER_FOLDS = 3
N_INNER_FOLDS = 3

RANDOM_SEARCH_ITER = 25
OPTUNA_N_TRIALS = 30

GA_POPULATION_SIZE = 16
GA_N_GENERATIONS = 10
GA_CX_PROB = 0.6
GA_MUT_PROB = 0.3
GA_MUT_INDPB = 0.2
GA_TOURNAMENT_SIZE = 3
GA_ELITE_SIZE = 2

DEFAULT_SCORING = "f1"
CORE_METRICS = ("accuracy", "precision", "recall", "f1", "roc_auc")

# Parametros del modelo SVM (fijos, no bajo busqueda): kernel RBF fijo
# (requisito del curso), probability=True (necesario para roc_auc),
# max_iter acotado (misma razon que en el proyecto local: evitar que el
# solver SMO tarde indefinidamente en converger en combinaciones
# patologicas de C/gamma -- ver models.py::_svm_factory).
SVM_MAX_ITER = 200_000
SVM_CACHE_SIZE_MIB = 2048  # generoso: T4 tiene 16GB: no es un hiperparametro bajo busqueda

print("Config OK")

## 8. Espacio de búsqueda de SVM (idéntico a `models.py::MODEL_REGISTRY["svm"]`)

`C` y `gamma`, ambos `float`, escala logarítmica. Reproduce
`HyperparamSpec`/`get_param_grid`/`get_param_distributions` de `models.py`
especializado a los 2 hiperparámetros de SVM (no hace falta el resto del
sistema genérico multi-modelo).

In [ ]:
from dataclasses import dataclass


@dataclass
class HyperparamSpec:
    name: str
    low: float
    high: float
    log: bool = True  # los dos hiperparametros de SVM (C, gamma) son log=True


SVM_HYPERPARAMS = [
    HyperparamSpec("C", low=1e-2, high=1e2),
    HyperparamSpec("gamma", low=1e-4, high=1e0),
]


def get_param_grid(hyperparams=SVM_HYPERPARAMS, n_points=4):
    '''Igual a models.py::get_param_grid especializado a floats log=True.
    n_points=4 (igual que localmente: numeric_hp_count=2 > 1, entonces
    n_points = max(2, min(4, int(200 ** (1/2)))) = 4 -> 4x4 = 16 combinaciones,
    igual que grid_search en la corrida local).'''
    grid = {}
    for hp in hyperparams:
        values = np.logspace(np.log10(hp.low), np.log10(hp.high), num=n_points)
        grid[hp.name] = list(values)
    return grid


def get_param_distributions(hyperparams=SVM_HYPERPARAMS):
    '''Igual a models.py::get_param_distributions para floats log=True.'''
    return {hp.name: loguniform(hp.low, hp.high) for hp in hyperparams}


SVM_PARAM_GRID = get_param_grid()
SVM_PARAM_DISTRIBUTIONS = get_param_distributions()
print("param_grid (n_points=4):")
for k, v in SVM_PARAM_GRID.items():
    print(f"  {k}: {[round(float(x), 6) for x in v]}")
print(f"Total combinaciones grid_search: {len(list(ParameterGrid(SVM_PARAM_GRID)))} (se espera 16)")

## 9. Preprocesamiento (idéntico a `preprocessing.py`, con salida densa)

Mismo `ColumnTransformer`: `OneHotEncoder(handle_unknown="ignore",
drop="if_binary")` en las categóricas + `StandardScaler()` en el resto
(SVM siempre tiene `requires_scaling=True`). Único cambio: `sparse_output=False`
(aviso #4 más arriba) — necesario para que cuML reciba un array denso.

In [ ]:
def build_column_transformer(categorical_columns=CATEGORICAL_COLUMNS):
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False),
                categorical_columns,
            )
        ],
        remainder=StandardScaler(),
    )

## 10. Balanceo (idéntico a `balancing.py`) + fábrica del modelo cuML

In [ ]:
BALANCING_TECHNIQUES = ("none", "smote", "adasyn", "class_weight")


def _safe_k_neighbors(y, default=5, cv_n_splits=None):
    '''Identico a balancing.py::_safe_k_neighbors.'''
    n_minority = int(pd.Series(y).value_counts().min())
    if cv_n_splits is not None and cv_n_splits > 1:
        n_minority = int(n_minority * (cv_n_splits - 1) / cv_n_splits)
    return max(1, min(default, n_minority - 1))


def get_resampler(technique, y_train=None, random_state=RANDOM_STATE, cv_n_splits=None):
    '''Identico a balancing.py::get_resampler.'''
    if technique in ("none", "class_weight"):
        return None
    if technique == "smote":
        k = _safe_k_neighbors(y_train, cv_n_splits=cv_n_splits) if y_train is not None else 5
        return SMOTE(random_state=random_state, k_neighbors=k)
    if technique == "adasyn":
        k = _safe_k_neighbors(y_train, cv_n_splits=cv_n_splits) if y_train is not None else 5
        return ADASYN(random_state=random_state, n_neighbors=k)
    raise ValueError(f"Tecnica de balanceo desconocida: '{technique}'")


def svm_factory(random_state=RANDOM_STATE, **kwargs):
    '''Equivalente cuML de models.py::_svm_factory. kernel='rbf' fijo,
    probability=True (necesario para roc_auc), max_iter/cache_size iguales
    en espiritu a la version CPU (ver aviso #5 sobre unidades de cache_size).'''
    return cuSVC(
        kernel="rbf",
        probability=True,
        max_iter=SVM_MAX_ITER,
        cache_size=SVM_CACHE_SIZE_MIB,
        random_state=random_state,
        output_type="numpy",
        **kwargs,
    )


def build_pipeline(technique, y_train=None, cv_n_splits=None, random_state=RANDOM_STATE, **model_kwargs):
    '''Identico a balancing.py::build_pipeline, especializado a SVM+cuML.'''
    steps = [("preprocessing", build_column_transformer())]

    resampler = get_resampler(technique, y_train=y_train, random_state=random_state, cv_n_splits=cv_n_splits)
    if resampler is not None:
        steps.append((technique, resampler))

    extra_kwargs = dict(model_kwargs)
    if technique == "class_weight":
        if y_train is None:
            raise ValueError("technique='class_weight' requiere y_train.")
        extra_kwargs["class_weight"] = "balanced"  # SVM: class_weight_param default ("balanced" nativo de SVC)

    estimator = svm_factory(random_state=random_state, **extra_kwargs)
    steps.append(("model", estimator))

    return ImbPipeline(steps=steps)  # sin memory=Memory(...): cache de joblib no aporta con GPU (el costo esta en el fit, no en recalcular SMOTE/ADASYN)


print("build_pipeline OK -- ejemplo:")
print(build_pipeline("smote", y_train=y, cv_n_splits=3, C=1.0, gamma=0.1))

## 11. Los 4 métodos de optimización (idénticos a `optimization.py`)

`grid_search`/`random_search`: implementación manual (ver nota metodológica
arriba) con `ParameterGrid`/`ParameterSampler` de scikit-learn — mismo
algoritmo que `GridSearchCV`/`RandomizedSearchCV` usan por dentro (por cada
candidato: CV interna -> promedio de folds -> se queda con el mejor ->
refit sobre todo el fold externo).

`bayesian_optuna`/`genetic_deap`: transcripción directa de
`run_bayesian_search`/`run_genetic_search` de `optimization.py` (mismo
sampler TPE, mismos operadores genéticos), con `n_jobs=1` siempre (una
sola GPU — no tiene sentido paralelizar evaluaciones independientes contra
el mismo contexto de GPU; esto es una decisión de implementación, no de
metodología, mismo principio que ya documenta el proyecto local en
`EXPERIMENT_STATUS.md` sección 8a: "no cambia folds/rangos/tamaños, solo
tiempo de pared").

In [ ]:
def _n_splits(cv):
    return int(cv.n_splits)


def _mean_cv_score(params, technique, X, y, cv, scoring):
    scorer = get_scorer(scoring)
    fold_scores = []
    for train_idx, val_idx in cv.split(X, y):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        pipe = build_pipeline(technique, y_train=y_tr, cv_n_splits=_n_splits(cv), random_state=RANDOM_STATE, **params)
        pipe.fit(X_tr, y_tr)
        fold_scores.append(scorer(pipe, X_val, y_val))
    return float(np.mean(fold_scores))


def _manual_search(candidates, technique, X, y, cv, scoring):
    '''Mismo algoritmo que GridSearchCV/RandomizedSearchCV: evalua cada
    candidato con CV interna, se queda con el de mejor score promedio
    (primer empate gana, igual que np.argmax), refit sobre todo (X, y).'''
    best_score, best_params = -np.inf, None
    for params in candidates:
        score = _mean_cv_score(params, technique, X, y, cv, scoring)
        if score > best_score:
            best_score, best_params = score, params

    best_pipeline = build_pipeline(technique, y_train=y, cv_n_splits=_n_splits(cv), random_state=RANDOM_STATE, **best_params)
    best_pipeline.fit(X, y)
    return {
        "best_params": best_params,
        "best_score": best_score,
        "best_pipeline": best_pipeline,
        "n_evaluations": len(candidates),
    }


def run_grid_search(technique, X, y, cv, scoring=DEFAULT_SCORING):
    candidates = list(ParameterGrid(SVM_PARAM_GRID))
    return _manual_search(candidates, technique, X, y, cv, scoring)


def run_random_search(technique, X, y, cv, scoring=DEFAULT_SCORING, n_iter=RANDOM_SEARCH_ITER):
    # ParameterSampler es EXACTAMENTE lo que RandomizedSearchCV usa por dentro
    # para generar candidatos -- misma secuencia dado el mismo random_state.
    candidates = list(ParameterSampler(SVM_PARAM_DISTRIBUTIONS, n_iter=n_iter, random_state=RANDOM_STATE))
    candidates = [{k: float(v) for k, v in c.items()} for c in candidates]
    return _manual_search(candidates, technique, X, y, cv, scoring)


def run_bayesian_search(technique, X, y, cv, scoring=DEFAULT_SCORING, n_trials=OPTUNA_N_TRIALS):
    '''Transcripcion de optimization.py::run_bayesian_search.'''
    scorer = get_scorer(scoring)

    def objective(trial):
        params = {
            "C": trial.suggest_float("C", 1e-2, 1e2, log=True),
            "gamma": trial.suggest_float("gamma", 1e-4, 1e0, log=True),
        }
        fold_scores = []
        for train_idx, val_idx in cv.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            pipe = build_pipeline(technique, y_train=y_tr, random_state=RANDOM_STATE, **params)
            pipe.fit(X_tr, y_tr)
            fold_scores.append(scorer(pipe, X_val, y_val))
        return float(np.mean(fold_scores))

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, n_jobs=1, show_progress_bar=False)

    best_params = study.best_params
    best_pipeline = build_pipeline(technique, y_train=y, cv_n_splits=_n_splits(cv), random_state=RANDOM_STATE, **best_params)
    best_pipeline.fit(X, y)
    return {
        "best_params": best_params,
        "best_score": float(study.best_value),
        "best_pipeline": best_pipeline,
        "n_evaluations": len(study.trials),
    }


# --- Algoritmo genetico (DEAP) -- transcripcion de optimization.py::run_genetic_search ---

def _init_gene(hp):
    return float(10 ** random.uniform(np.log10(hp.low), np.log10(hp.high)))


def _cx_uniform(ind1, ind2, indpb=0.5):
    for i in range(len(ind1)):
        if random.random() < indpb:
            ind1[i], ind2[i] = ind2[i], ind1[i]
    return ind1, ind2


def _mutate_individual(individual, hyperparams, indpb, sigma=0.15):
    for i, hp in enumerate(hyperparams):
        if random.random() > indpb:
            continue
        log_lo, log_hi = np.log10(hp.low), np.log10(hp.high)
        current_log = np.log10(max(individual[i], 1e-300))
        new_log = current_log + random.gauss(0, sigma * (log_hi - log_lo))
        new_log = min(max(new_log, log_lo), log_hi)
        individual[i] = float(10 ** new_log)
    return (individual,)


def run_genetic_search(
    technique, X, y, cv, scoring=DEFAULT_SCORING,
    population_size=GA_POPULATION_SIZE, n_generations=GA_N_GENERATIONS,
    cx_prob=GA_CX_PROB, mut_prob=GA_MUT_PROB, mut_indpb=GA_MUT_INDPB,
    tournament_size=GA_TOURNAMENT_SIZE, elite_size=GA_ELITE_SIZE,
):
    random.seed(RANDOM_STATE)  # DEAP usa el modulo `random` global, igual que localmente

    if not hasattr(creator, "FitnessMax"):
        creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    if not hasattr(creator, "Individual"):
        creator.create("Individual", list, fitness=creator.FitnessMax)

    hyperparams = SVM_HYPERPARAMS
    scorer = get_scorer(scoring)

    def _init_individual():
        return creator.Individual([_init_gene(hp) for hp in hyperparams])

    def _decode(individual):
        return {hp.name: gene for hp, gene in zip(hyperparams, individual)}

    def _evaluate(individual):
        params = _decode(individual)
        fold_scores = []
        for train_idx, val_idx in cv.split(X, y):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
            pipe = build_pipeline(technique, y_train=y_tr, random_state=RANDOM_STATE, **params)
            pipe.fit(X_tr, y_tr)
            fold_scores.append(scorer(pipe, X_val, y_val))
        return (float(np.mean(fold_scores)),)

    toolbox = base.Toolbox()
    toolbox.register("individual", _init_individual)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("mate", _cx_uniform, indpb=0.5)
    toolbox.register("mutate", _mutate_individual, hyperparams=hyperparams, indpb=mut_indpb)
    toolbox.register("select", tools.selTournament, tournsize=tournament_size)
    toolbox.register("evaluate", _evaluate)

    population = toolbox.population(n=population_size)
    for ind, fit in zip(population, map(toolbox.evaluate, population)):
        ind.fitness.values = fit
    n_evaluations = len(population)

    for _generation in range(1, n_generations + 1):
        elite = [toolbox.clone(ind) for ind in tools.selBest(population, elite_size)]
        offspring = [toolbox.clone(ind) for ind in toolbox.select(population, len(population) - elite_size)]

        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < cx_prob:
                toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < mut_prob:
                toolbox.mutate(mutant)
                del mutant.fitness.values

        invalid = [ind for ind in offspring if not ind.fitness.valid]
        for ind, fit in zip(invalid, map(toolbox.evaluate, invalid)):
            ind.fitness.values = fit
        n_evaluations += len(invalid)

        population = elite + offspring

    best_ind = tools.selBest(population, 1)[0]
    best_params = _decode(best_ind)
    best_pipeline = build_pipeline(technique, y_train=y, cv_n_splits=_n_splits(cv), random_state=RANDOM_STATE, **best_params)
    best_pipeline.fit(X, y)
    return {
        "best_params": best_params,
        "best_score": float(best_ind.fitness.values[0]),
        "best_pipeline": best_pipeline,
        "n_evaluations": n_evaluations,
    }


OPTIMIZATION_DISPATCH = {
    "grid_search": run_grid_search,
    "random_search": run_random_search,
    "bayesian_optuna": run_bayesian_search,
    "genetic_deap": run_genetic_search,
}
print("4 metodos de optimizacion definidos:", list(OPTIMIZATION_DISPATCH.keys()))

## 12. Validación cruzada anidada (idéntica a `nested_cv.py`)

In [ ]:
def compute_core_metrics(y_true, y_pred, y_proba):
    '''Identico a nested_cv.py::compute_core_metrics.'''
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_proba)),
    }


def run_nested_cv(technique, method, X, y, n_outer_folds=N_OUTER_FOLDS, n_inner_folds=N_INNER_FOLDS,
                   scoring=DEFAULT_SCORING, random_state=RANDOM_STATE):
    '''Identico a nested_cv.py::run_nested_cv, especializado a SVM+cuML.'''
    outer_cv = StratifiedKFold(n_splits=n_outer_folds, shuffle=True, random_state=random_state)
    inner_cv = StratifiedKFold(n_splits=n_inner_folds, shuffle=True, random_state=random_state)

    fold_results = []
    t_start_total = time.time()

    for fold_index, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        X_train_outer, X_test_outer = X.iloc[train_idx], X.iloc[test_idx]
        y_train_outer, y_test_outer = y.iloc[train_idx], y.iloc[test_idx]

        t0 = time.time()
        opt_result = OPTIMIZATION_DISPATCH[method](technique, X_train_outer, y_train_outer, inner_cv, scoring=scoring)
        fit_time_seconds = time.time() - t0

        y_pred = opt_result["best_pipeline"].predict(X_test_outer)
        y_proba = opt_result["best_pipeline"].predict_proba(X_test_outer)[:, 1]
        y_true = y_test_outer.to_numpy()

        fold_results.append({
            "fold_index": fold_index,
            "best_params": opt_result["best_params"],
            "metrics": compute_core_metrics(y_true, y_pred, y_proba),
            "fit_time_seconds": fit_time_seconds,
        })
        print(
            f"    fold {fold_index}: f1={fold_results[-1]['metrics']['f1']:.4f} "
            f"best_params={ {k: round(v, 6) for k, v in opt_result['best_params'].items()} } "
            f"({fit_time_seconds:.1f}s)"
        )

    total_time_seconds = time.time() - t_start_total

    row = {
        "model": "svm",
        "technique": technique,
        "method": method,
        "total_time_seconds": total_time_seconds,
        "n_outer_folds": len(fold_results),
    }
    for metric_name in CORE_METRICS:
        values = [fr["metrics"][metric_name] for fr in fold_results]
        row[f"{metric_name}_mean"] = float(np.mean(values))
        row[f"{metric_name}_std"] = float(np.std(values))
        row[f"{metric_name}_per_fold"] = json.dumps(values)
    row["best_params_per_fold"] = json.dumps([fr["best_params"] for fr in fold_results])
    row["random_state"] = random_state
    row["n_inner_folds"] = n_inner_folds
    return row


print("run_nested_cv definido")

## 13. Selección de combinaciones a correr

**Ya corridas localmente (CPU, en `results/experiments_master.csv`)** —
verificado el 2026-09-23: `svm/none/grid_search`, `svm/none/random_search`,
`svm/none/bayesian_optuna` (3/16).

**Pendientes (13/16)** — todas las demás. `svm/none/genetic_deap` puede
estar corriendo AHORA MISMO en local (proceso CPU en curso al momento de
armar este notebook) — si ya terminó localmente antes de correr esta
celda, sacarla de `COMBOS_TO_RUN` para no repetir trabajo.

Editar `COMBOS_TO_RUN` para correr solo un subconjunto (ej. dejar solo
`smote`/`adasyn`, que son las que más tardaron/se trabaron en CPU local
según `EXPERIMENT_STATUS.md`).

In [ ]:
ALL_SVM_COMBOS = [(t, m) for t in BALANCING_TECHNIQUES
                  for m in ("grid_search", "random_search", "bayesian_optuna", "genetic_deap")]
assert len(ALL_SVM_COMBOS) == 16

DONE_LOCALLY = [
    ("none", "grid_search"),
    ("none", "random_search"),
    ("none", "bayesian_optuna"),
]

PENDING = [c for c in ALL_SVM_COMBOS if c not in DONE_LOCALLY]
assert len(PENDING) == 13

# Editar esta lista para correr solo un subconjunto:
COMBOS_TO_RUN = list(PENDING)

print(f"Ya corridas localmente: {DONE_LOCALLY}")
print(f"Pendientes ({len(PENDING)}): {PENDING}")
print(f"COMBOS_TO_RUN ({len(COMBOS_TO_RUN)}): {COMBOS_TO_RUN}")

## 14. Ejecutar — guardado incremental (una fila apenas termina cada combinación)

Si Colab se desconecta a mitad de camino: volver a correr esta celda (no
hace falta repetir nada de arriba salvo reinstalar/reimportar si se
reinició la sesión) — salta automáticamente las combinaciones que ya estén
en `RESULTS_CSV_PATH`.

In [ ]:
RESULTS_CSV_PATH = Path("svm_gpu_results.csv")
ERRORS_LOG_PATH = Path("svm_gpu_errors_log.csv")

already_done = set()
if RESULTS_CSV_PATH.exists():
    existing = pd.read_csv(RESULTS_CSV_PATH)
    already_done = set(zip(existing["technique"], existing["method"]))
    print(f"{len(already_done)} combinaciones ya en {RESULTS_CSV_PATH}, se omiten.")

to_run = [c for c in COMBOS_TO_RUN if c not in already_done]
print(f"{len(to_run)} combinaciones a correr en esta sesion: {to_run}")

for technique, method in to_run:
    print(f"\n=== svm / {technique} / {method} ===")
    t0 = time.time()
    try:
        row = run_nested_cv(technique, method, X, y)
    except Exception as exc:
        elapsed = time.time() - t0
        print(f"  ERROR tras {elapsed:.1f}s: {exc!r}")
        traceback.print_exc()
        err_row = pd.DataFrame([{
            "technique": technique, "method": method,
            "elapsed_seconds": elapsed, "error": repr(exc), "traceback": traceback.format_exc(),
        }])
        err_row.to_csv(ERRORS_LOG_PATH, mode="a", header=not ERRORS_LOG_PATH.exists(), index=False)
        continue

    row_df = pd.DataFrame([row])
    row_df.to_csv(RESULTS_CSV_PATH, mode="a", header=not RESULTS_CSV_PATH.exists(), index=False)
    print(
        f"  OK svm/{technique}/{method} -- "
        f"f1={row['f1_mean']:.4f}+/-{row['f1_std']:.4f} "
        f"roc_auc={row['roc_auc_mean']:.4f}+/-{row['roc_auc_std']:.4f} "
        f"({row['total_time_seconds']:.1f}s)"
    )

print("\nListo. Filas guardadas en:", RESULTS_CSV_PATH.resolve())

## 15. Resultado final + descarga

Mismo esquema exacto de columnas que `results/experiments_master.csv`
local (`model, technique, method, total_time_seconds, n_outer_folds,
accuracy_mean, accuracy_std, accuracy_per_fold, precision_mean,
precision_std, precision_per_fold, recall_mean, recall_std,
recall_per_fold, f1_mean, f1_std, f1_per_fold, roc_auc_mean, roc_auc_std,
roc_auc_per_fold, best_params_per_fold, random_state, n_inner_folds`) — se
puede concatenar directamente (`pd.concat`) o pegar las filas sin
reformatear.

In [ ]:
final_results = pd.read_csv(RESULTS_CSV_PATH) if RESULTS_CSV_PATH.exists() else pd.DataFrame()
display(final_results)

if ERRORS_LOG_PATH.exists():
    print("\nHubo errores en algunas combinaciones -- revisar antes de dar por completo el batch:")
    display(pd.read_csv(ERRORS_LOG_PATH)[["technique", "method", "elapsed_seconds", "error"]])

from google.colab import files
files.download(str(RESULTS_CSV_PATH))